# Train Both Agents — Colab Pro+

Trains **two agents** back-to-back in a single session:

| Agent | Chunk file | Purpose |
|---|---|---|
| **Automobile Agent** (Agent 1) | `automobile_agent/chunks_clean.jsonl` | Go4Garage + auto industry |
| **JT Agent** (Jay's Agent) | `jt_agent/chunks_clean.jsonl` | Personal life, device, profile |

Both use the same Qwen 2.5-3B-Instruct fine-tune pipeline with identical quality settings.

> **Recommended runtime:** A100 40GB. Go to Runtime → Change runtime type → A100.

## Step 1 — Check GPU

In [ ]:
!nvidia-smi

## Step 2 — Mount Google Drive

Checkpoints and exported models are saved to Drive so a recycled session does not lose progress.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 3 — Install dependencies

In [ ]:
!pip install -q "transformers>=4.45" "datasets>=2.20" "accelerate>=0.30" \
               "trl>=0.10" "bitsandbytes>=0.43" "tiktoken>=0.7" "pyyaml>=6.0"

## Step 4 — Upload agent package + both chunk files

Run the cell below and upload:
- `personal_folder_agent_pkg.zip` (the agent code from your machine)
- `automobile_agent/chunks_clean.jsonl` → rename to **`chunks_auto.jsonl`** before uploading
- `jt_agent/chunks_clean.jsonl` → rename to **`chunks_jt.jsonl`** before uploading

Or copy them to Drive first using rclone on your Windows machine:
```powershell
rclone copy "C:\Users\abcom\jpbir\runs\automobile_agent\chunks_clean.jsonl" "gdrive:personal-folder-agent/chunks_auto.jsonl"
rclone copy "C:\Users\abcom\jpbir\runs\jt_agent\chunks_clean.jsonl" "gdrive:personal-folder-agent/chunks_jt.jsonl"
```

In [ ]:
import os, zipfile
from google.colab import files

PROJECT_DIR = '/content/drive/MyDrive/personal-folder-agent'
os.makedirs(PROJECT_DIR, exist_ok=True)

uploaded = files.upload()

if 'personal_folder_agent_pkg.zip' in uploaded:
    with zipfile.ZipFile('personal_folder_agent_pkg.zip') as z:
        z.extractall(PROJECT_DIR)
    print('Agent package extracted.')

for fname in uploaded:
    if fname.startswith('chunks_') and fname.endswith('.jsonl'):
        dest = os.path.join(PROJECT_DIR, fname)
        os.rename(fname, dest)
        size_mb = os.path.getsize(dest) / 1e6
        print(f'Saved {fname} -> {dest}  ({size_mb:.0f} MB)')

In [ ]:
import sys
PROJECT_DIR = '/content/drive/MyDrive/personal-folder-agent'
sys.path.insert(0, PROJECT_DIR)

import personal_folder_agent
print('package loaded from', personal_folder_agent.__file__)

## Step 5 — Confirm environment

In [ ]:
from personal_folder_agent.utils.environment import detect_environment, recommend_settings

environment = detect_environment()
print(environment.describe())
print('colab tier hint:', environment.colab_tier_hint)
print('RAM (GB):', round(environment.total_ram_gb or 0, 1))
for note in environment.detection_notes:
    print(' -', note)

---
# AGENT 1 — Automobile Agent

Go4Garage + auto industry corpus (~102k chunks, 315 MB)

## A1-1. Load Automobile chunks

In [ ]:
from personal_folder_agent.utils.chunk_io import load_chunks_jsonl

AUTO_CHUNKS_FILE = f'{PROJECT_DIR}/chunks_auto.jsonl'

chunks_auto = load_chunks_jsonl(AUTO_CHUNKS_FILE)
total_tokens_auto = sum(c.token_count for c in chunks_auto)
print(f'Automobile Agent: {len(chunks_auto):,} chunks, {total_tokens_auto:,} tokens')

## A1-2. Configure and train Automobile Agent

In [ ]:
from personal_folder_agent.components.training_pipeline import TrainingConfig, QwenTrainingPipeline

config_auto = TrainingConfig(
    model='Qwen2.5-3B-Instruct',
    epochs=3,
    batch_size=2,
    learning_rate=2e-5,
    gradient_accumulation_steps=4,
    max_length=1024,
    checkpoint_dir=f'{PROJECT_DIR}/checkpoints_auto',
    output_dir=f'{PROJECT_DIR}/output_auto',
    auto_tune_hardware=True,
)

pipeline_auto = QwenTrainingPipeline(config_auto)
metadata_auto = pipeline_auto.train(chunks_auto)

summary_auto = pipeline_auto.run_summary
print()
print('=== Automobile Agent Training Complete ===')
print('final training loss  :', round(metadata_auto.performance_metrics.final_loss, 4))
print('final validation loss:', metadata_auto.performance_metrics.final_validation_loss)
print('final accuracy       :', round(metadata_auto.performance_metrics.final_accuracy, 4))

## A1-3. Resume if session was recycled mid-run

In [ ]:
# Only run this cell if training was interrupted.
from personal_folder_agent.components.training_pipeline import find_latest_checkpoint

latest_auto = find_latest_checkpoint(config_auto.checkpoint_dir)
print('resuming from', latest_auto)

metadata_auto = pipeline_auto.resume_from_checkpoint(config_auto.checkpoint_dir)
print('final validation loss:', metadata_auto.performance_metrics.final_validation_loss)

## A1-4. Export Automobile Agent model

In [ ]:
from personal_folder_agent.components.model_exporter import ExportConfig, LocalModelExporter

export_config_auto = ExportConfig(
    output_dir=config_auto.output_dir,
    version='1.0.0',
    folder_list=sorted({c.source_file.rsplit('/', 1)[0] for c in chunks_auto}),
)

exporter_auto = LocalModelExporter(export_config_auto)
export_result_auto = exporter_auto.export(metadata_auto)
print('Automobile Agent exported to:', export_result_auto.local_path)

---
# AGENT 2 — JT Agent (Jay's Agent)

Personal life, device data, profile corpus (~297k chunks, 928 MB)

> If you ran the `merge_gz_parts.py` script locally, this file also includes
> `jayti_personal_chunks.jsonl.gz` (~266k) and `diac_case_chunks.jsonl.gz` (227).

## A2-1. Load JT chunks

In [ ]:
from personal_folder_agent.utils.chunk_io import load_chunks_jsonl

JT_CHUNKS_FILE = f'{PROJECT_DIR}/chunks_jt.jsonl'

chunks_jt = load_chunks_jsonl(JT_CHUNKS_FILE)
total_tokens_jt = sum(c.token_count for c in chunks_jt)
print(f'JT Agent: {len(chunks_jt):,} chunks, {total_tokens_jt:,} tokens')

## A2-2. Configure and train JT Agent

In [ ]:
from personal_folder_agent.components.training_pipeline import TrainingConfig, QwenTrainingPipeline

config_jt = TrainingConfig(
    model='Qwen2.5-3B-Instruct',
    epochs=3,
    batch_size=2,
    learning_rate=2e-5,
    gradient_accumulation_steps=4,
    max_length=1024,
    checkpoint_dir=f'{PROJECT_DIR}/checkpoints_jt',
    output_dir=f'{PROJECT_DIR}/output_jt',
    auto_tune_hardware=True,
)

pipeline_jt = QwenTrainingPipeline(config_jt)
metadata_jt = pipeline_jt.train(chunks_jt)

summary_jt = pipeline_jt.run_summary
print()
print('=== JT Agent Training Complete ===')
print('final training loss  :', round(metadata_jt.performance_metrics.final_loss, 4))
print('final validation loss:', metadata_jt.performance_metrics.final_validation_loss)
print('final accuracy       :', round(metadata_jt.performance_metrics.final_accuracy, 4))

## A2-3. Resume if session was recycled mid-run

In [ ]:
# Only run this cell if training was interrupted.
from personal_folder_agent.components.training_pipeline import find_latest_checkpoint

latest_jt = find_latest_checkpoint(config_jt.checkpoint_dir)
print('resuming from', latest_jt)

metadata_jt = pipeline_jt.resume_from_checkpoint(config_jt.checkpoint_dir)
print('final validation loss:', metadata_jt.performance_metrics.final_validation_loss)

## A2-4. Export JT Agent model

In [ ]:
from personal_folder_agent.components.model_exporter import ExportConfig, LocalModelExporter

export_config_jt = ExportConfig(
    output_dir=config_jt.output_dir,
    version='1.0.0',
    folder_list=sorted({c.source_file.rsplit('/', 1)[0] for c in chunks_jt}),
)

exporter_jt = LocalModelExporter(export_config_jt)
export_result_jt = exporter_jt.export(metadata_jt)
print('JT Agent exported to:', export_result_jt.local_path)

---
# Summary

In [ ]:
print('=' * 50)
print('TRAINING COMPLETE')
print('=' * 50)
print()
print('Automobile Agent')
print(f'  Chunks : {len(chunks_auto):,}')
print(f'  Loss   : {round(metadata_auto.performance_metrics.final_loss, 4)}')
print(f'  Output : {export_result_auto.local_path}')
print()
print('JT Agent')
print(f'  Chunks : {len(chunks_jt):,}')
print(f'  Loss   : {round(metadata_jt.performance_metrics.final_loss, 4)}')
print(f'  Output : {export_result_jt.local_path}')

---
# Optional — Chat with each agent

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

def chat(model_path, question):
    tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-3B-Instruct', trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.float16,
        device_map='auto',
        trust_remote_code=True,
    )
    inputs = tokenizer(question, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=200, temperature=0.7, do_sample=True)
    return tokenizer.decode(out[0], skip_special_tokens=True)

# Test Automobile Agent
answer = chat(export_result_auto.local_path, 'What products does Go4Garage offer?')
print('[Automobile Agent]', answer)

In [ ]:
# Test JT Agent
answer = chat(export_result_jt.local_path, 'Tell me about Jayathi.')
print('[JT Agent]', answer)